# TRANCE Framework — Training Pipeline v3
### 30-Day Hospital Readmission Prediction (MIMIC-IV)

**Pipeline overview:**
1. Mount Google Drive & set paths
2. Install dependencies
3. Load & fuse features + embeddings
4. SMOTETomek class imbalance handling
5. Optuna Bayesian HPO (LightGBM)
6. Patient-level GroupKFold cross-validation
7. LightGBM + XGBoost ensemble + meta-learner stacking
8. Isotonic calibration
9. Threshold optimisation (recall ≥ 80%)
10. SHAP interpretability + plots
11. Save model → Google Drive

> **Expected AUROC:** 0.82–0.86 with full dataset + T5 embeddings  
> **Runtime:** ~5–8 hours on Colab T4 GPU (50 Optuna trials, 546k rows)

## Cell 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── Set your project root here ─────────────────────────────────────────────
PROJECT_ROOT = '/content/drive/MyDrive/readmission-ai'   # <-- CHANGE THIS

import os, sys
os.makedirs(PROJECT_ROOT, exist_ok=True)
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))
print(f"Project root: {PROJECT_ROOT}")
print(f"GPU available: ", end="")
import subprocess
r = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                  capture_output=True, text=True)
print(r.stdout.strip() if r.returncode == 0 else "CPU only")

## Cell 2 — Install Dependencies

In [ ]:
%%capture
!pip install lightgbm xgboost optuna shap imbalanced-learn scikit-learn pandas numpy matplotlib joblib

# Verify GPU for XGBoost
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True)
HAS_GPU = result.returncode == 0
print(f"GPU: {'YES' if HAS_GPU else 'NO (CPU mode)'}")

import lightgbm as lgb, xgboost as xgb, optuna, shap
print(f"LightGBM: {lgb.__version__}")
print(f"XGBoost:  {xgb.__version__}")
print(f"Optuna:   {optuna.__version__}")
print(f"SHAP:     {shap.__version__}")

## Cell 3 — Configuration
Set paths and tunable constants. Edit these before running.

In [ ]:
import os

# ── Paths (all relative to PROJECT_ROOT) ──────────────────────────────────
DATA_DIR    = os.path.join(PROJECT_ROOT, 'data')
MODELS_DIR  = os.path.join(PROJECT_ROOT, 'models')
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results')
FIGURES_DIR = os.path.join(PROJECT_ROOT, 'figures')

FEATURES_CSV   = os.path.join(DATA_DIR, 'ultimate_features_pruned.csv')   # from 01b
EMBEDDINGS_CSV = os.path.join(DATA_DIR, 'clinical_t5_embeddings.csv')     # from 02
MAIN_MODEL_PKL = os.path.join(MODELS_DIR, 'trance_framework.pkl')
SELECTED_FEATURES_JSON = os.path.join(MODELS_DIR, 'selected_features.json')

for d in [DATA_DIR, MODELS_DIR, RESULTS_DIR, FIGURES_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Tunable constants ──────────────────────────────────────────────────────
RANDOM_STATE  = 42
OPTUNA_TRIALS = 50     # ~5-8 hrs on T4. Use 10 for a quick smoke test.
N_FOLDS       = 5      # patient-level GroupKFold folds
DART_MAX_TREES = 800   # cap DART trees (no early stopping)
ENABLE_STACK  = True   # LightGBM + XGBoost meta-learner
ENABLE_SMOTE  = True   # SMOTETomek oversampling
TEST_FRAC     = 0.15   # fraction of patients held out as test
VAL_FRAC      = 0.15   # fraction of patients held out as validation

print("Configuration:")
print(f"  FEATURES_CSV   : {FEATURES_CSV}")
print(f"  EMBEDDINGS_CSV : {EMBEDDINGS_CSV}")
print(f"  OPTUNA_TRIALS  : {OPTUNA_TRIALS}")
print(f"  Features exist : {os.path.exists(FEATURES_CSV)}")
print(f"  Embeddings exist: {os.path.exists(EMBEDDINGS_CSV)}")

## Cell 4 — Imports

In [ ]:
import gc
import json
import logging
import warnings
from datetime import datetime
from typing import Dict, List, Optional, Tuple

import joblib
import lightgbm as lgb
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import shap
from sklearn.calibration import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, brier_score_loss, classification_report,
    f1_score, log_loss, precision_recall_curve, roc_auc_score, roc_curve,
)
from sklearn.model_selection import GroupKFold

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(os.path.join(PROJECT_ROOT, 'train.log'),
                           mode='w', encoding='utf-8'),
    ]
)
logger = logging.getLogger(__name__)

def _optuna_callback(study, trial):
    if trial.value is None: return
    logger.info(
        '  Trial %3d | AUROC: %.4f | best: %.4f | boosting: %-5s | leaves: %s | lr: %.4f',
        trial.number, trial.value, study.best_value,
        trial.params.get('boosting_type', '?'),
        trial.params.get('num_leaves', '?'),
        trial.params.get('learning_rate', float('nan')),
    )

print("Imports complete.")

## Cell 5 — Load & Fuse Data
Loads `ultimate_features_pruned.csv` + `clinical_t5_embeddings.csv` and merges on `hadm_id`.

In [ ]:
def load_data() -> Tuple[pd.DataFrame, pd.Series, pd.Series]:
    """
    Loads structured features + optional ClinicalT5 embeddings.

    Returns
    -------
    X      : pd.DataFrame  — feature matrix (admissions × features)
    y      : pd.Series     — binary target (readmit_30)
    groups : pd.Series     — subject_id, used for patient-level GroupKFold
    """
    for d in [RESULTS_DIR, FIGURES_DIR, MODELS_DIR]:
        os.makedirs(d, exist_ok=True)

    pruned = FEATURES_CSV.replace(".csv", "_pruned.csv")
    path   = pruned if os.path.exists(pruned) else FEATURES_CSV
    logger.info("Loading features: %s", path)
    tab = pd.read_csv(path, low_memory=False).fillna(0)

    selected: Optional[List[str]] = None
    if os.path.exists(SELECTED_FEATURES_JSON):
        with open(SELECTED_FEATURES_JSON) as f:
            selected = json.load(f)
        logger.info("Selected features: %d", len(selected))

    if os.path.exists(EMBEDDINGS_CSV):
        emb = pd.read_csv(EMBEDDINGS_CSV, low_memory=False)
        df  = tab.merge(emb, on="hadm_id", how="left").fillna(0)
        logger.info("Fused shape: %s", df.shape)
    else:
        df = tab.copy()
        logger.warning("Embeddings not found — running tabular-only mode.")

    groups  = df["subject_id"].astype(int)
    y       = df["readmit_30"].astype("int8")
    id_cols = {"subject_id", "hadm_id", "readmit_30"}

    if selected:
        emb_cols = [c for c in df.columns if c.startswith("ct5_")]
        keep = [c for c in selected if c in df.columns] + emb_cols
        X = df[keep].copy()
    else:
        X = df.drop(columns=list(id_cols & set(df.columns)), errors="ignore")

    logger.info("Feature matrix: %s | Positive rate: %.2f%%", X.shape, y.mean() * 100)
    return X, y, groups


In [ ]:
X, y, groups = load_data()
print(f"X shape: {X.shape}")
print(f"Positive rate: {y.mean()*100:.2f}%")
print(f"Sample features: {list(X.columns[:5])}")

## Cell 6 — Class Imbalance: SMOTETomek
Oversamples minority class (readmit=1) to 35% ratio. Applied **only to training fold** — val and test are never touched.

In [ ]:
def apply_smote(X_tr: np.ndarray, y_tr: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """
    SMOTETomek = SMOTE oversampling of minority class + Tomek link cleaning.
    Applied ONLY to training data — validation and test are never touched.

    Install dependency:  pip install imbalanced-learn
    Set ENABLE_SMOTE=False above to skip.
    """
    try:
        from imblearn.combine import SMOTETomek
        from imblearn.over_sampling import SMOTE

        ratio = (y_tr == 1).sum() / max((y_tr == 0).sum(), 1)
        if ratio > 0.40:
            logger.info("Class ratio %.2f already reasonable — skipping SMOTE.", ratio)
            return X_tr, y_tr

        logger.info(
            "Applying SMOTETomek (before: %d pos / %d neg, ratio=%.3f) ...",
            (y_tr == 1).sum(), (y_tr == 0).sum(), ratio,
        )
        sm = SMOTETomek(
            smote=SMOTE(
                sampling_strategy=0.35,  # oversample to 35% minority
                random_state=RANDOM_STATE,
                # n_jobs removed: not supported in older imbalanced-learn versions
            ),
            random_state=RANDOM_STATE,
            # n_jobs removed from SMOTETomek: use fit_resample instead
        )
        X_res, y_res = sm.fit_resample(X_tr, y_tr)
        logger.info(
            "After SMOTETomek: %d pos / %d neg (ratio=%.3f)",
            (y_res == 1).sum(), (y_res == 0).sum(),
            (y_res == 1).sum() / max((y_res == 0).sum(), 1),
        )
        return X_res, y_res

    except ImportError:
        logger.warning(
            "imbalanced-learn not installed — skipping SMOTE. "
            "Install with: pip install imbalanced-learn"
        )
        return X_tr, y_tr
    except Exception as e:
        logger.warning("SMOTE failed (%s) — continuing without.", e)
        return X_tr, y_tr


## Cell 7 — Temporal Patient-Level Split
Sorts patients by first admission date (temporal ordering). Last 15% = test, prior 15% = val. No patient appears in multiple splits.

In [ ]:
# Temporal patient-level split
pat_first   = groups.reset_index(drop=True).groupby(groups.values).first()
sorted_pats = pat_first.sort_values().index.tolist()

n_test  = int(len(sorted_pats) * TEST_FRAC)
n_val   = int(len(sorted_pats) * VAL_FRAC)

test_pats  = set(sorted_pats[-n_test:])
val_pats   = set(sorted_pats[-(n_test + n_val):-n_test])
train_pats = set(sorted_pats[:-(n_test + n_val)])

train_mask = groups.isin(train_pats).values
val_mask   = groups.isin(val_pats).values
test_mask  = groups.isin(test_pats).values

X_tr  = X[train_mask].values.astype(np.float32); y_tr = y[train_mask].values
X_val = X[val_mask].values.astype(np.float32);   y_val = y[val_mask].values
X_te  = X[test_mask].values.astype(np.float32);  y_te  = y[test_mask].values
features = list(X.columns)

logger.info("Train: %d | Val: %d | Test: %d", len(y_tr), len(y_val), len(y_te))
logger.info("Train readmit: %.2f%% | Val: %.2f%% | Test: %.2f%%",
            y_tr.mean()*100, y_val.mean()*100, y_te.mean()*100)

pos_weight = float((y_tr == 0).sum() / max((y_tr == 1).sum(), 1))
logger.info("Class imbalance (pos_weight): %.2f", pos_weight)

# Apply SMOTE to training fold only
if ENABLE_SMOTE:
    X_tr, y_tr = apply_smote(X_tr, y_tr)

## Cell 8 — Optuna Bayesian HPO
Searches LightGBM hyperparameters + imbalance strategy over `OPTUNA_TRIALS` trials.

> **Tip:** Set `OPTUNA_TRIALS = 10` for a quick 30-min smoke test before the full run.

In [ ]:
def optimize_lgbm(
    X_tr: np.ndarray, y_tr: np.ndarray,
    X_val: np.ndarray, y_val: np.ndarray,
    pos_weight: float,
    n_trials: int = OPTUNA_TRIALS,
) -> Dict:
    """
    Bayesian HPO via Optuna (TPE sampler, Hyperband pruner).
    Searches: boosting type, tree structure, regularisation, imbalance strategy.

    Key imbalance fix:
      - GOSS boosting cannot use bagging (subsample/subsample_freq are removed)
      - Optuna picks between scale_pos_weight and is_unbalance per trial
    """
    logger.info("Optuna HPO: %d trials ...", n_trials)

    def objective(trial: optuna.Trial) -> float:
        # gbdt/goss appear twice to reduce costly dart sampling frequency
        boosting  = trial.suggest_categorical("boosting_type", ["gbdt", "gbdt", "dart", "goss", "goss"])
        imbalance = trial.suggest_categorical(
            "imbalance_strategy", ["scale_pos_weight", "is_unbalance"]
        )

        params: Dict = {
            "objective":         "binary",
            "metric":            "auc",
            "verbosity":         -1,
            "boosting_type":     boosting,
            "num_leaves":        trial.suggest_int("num_leaves", 63, 300),
            "max_depth":         trial.suggest_int("max_depth", 6, 16),
            "learning_rate":     trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
            # DART has no early stopping so cap trees to avoid 20-min trials
            "n_estimators": (
                trial.suggest_int("n_estimators", 500, DART_MAX_TREES)
                if boosting == "dart"
                else trial.suggest_int("n_estimators", 500, 3000)
            ),
            "min_child_samples": trial.suggest_int("min_child_samples", 5, 50),
            "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.4, 1.0),
            "colsample_bynode":  trial.suggest_float("colsample_bynode", 0.4, 1.0),
            "reg_alpha":         trial.suggest_float("reg_alpha", 0.0, 2.0),
            "reg_lambda":        trial.suggest_float("reg_lambda", 0.0, 2.0),
            "min_split_gain":    trial.suggest_float("min_split_gain", 0.0, 1.0),
            "path_smooth":       trial.suggest_float("path_smooth", 0.0, 1.0),
            "random_state":      RANDOM_STATE,
            "n_jobs":            -1,
        }

        # -- GOSS fix: remove bagging params (incompatible with GOSS) ------
        if boosting in ("gbdt", "dart"):
            params["subsample"]      = trial.suggest_float("subsample", 0.5, 1.0)
            params["subsample_freq"] = trial.suggest_int("subsample_freq", 1, 5)
        if boosting == "dart":
            params["drop_rate"] = trial.suggest_float("drop_rate", 0.05, 0.4)
        if boosting == "goss":
            params["top_rate"]   = trial.suggest_float("top_rate", 0.05, 0.4)
            params["other_rate"] = trial.suggest_float("other_rate", 0.05, 0.2)

        # -- Imbalance strategy: mutually exclusive -------------------------
        if imbalance == "scale_pos_weight":
            params["scale_pos_weight"] = pos_weight
        else:
            params["is_unbalance"] = True

        m = lgb.LGBMClassifier(**params)
        m.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(50, verbose=False),
                       lgb.log_evaluation(-1)],
        )
        return roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])

    sampler = optuna.samplers.TPESampler(multivariate=True, seed=RANDOM_STATE)
    # MedianPruner is safe and reliable; HyperbandPruner can terminate early unexpectedly
    pruner  = optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=5)
    study   = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner)
    logger.info("Starting Optuna optimization: %d trials", n_trials)
    study.optimize(
        objective,
        n_trials=n_trials,
        show_progress_bar=False,
        callbacks=[_optuna_callback],
        catch=(Exception,),  # log failed trials but don't abort the whole study
    )
    logger.info("Optuna complete. %d/%d trials succeeded.", 
                len([t for t in study.trials if t.value is not None]), n_trials)

    best = study.best_params.copy()
    # Remove Optuna-only key before passing params to LightGBM
    imbalance_strategy = best.pop("imbalance_strategy", "scale_pos_weight")
    if imbalance_strategy == "scale_pos_weight":
        best["scale_pos_weight"] = pos_weight
    else:
        best["is_unbalance"] = True

    best.update({
        "objective": "binary", "metric": "auc",
        "verbosity": -1, "n_jobs": -1, "random_state": RANDOM_STATE,
    })
    logger.info("Best HPO AUROC: %.4f", study.best_value)
    logger.info("Best params: %s", best)
    return best


In [ ]:
best_params = optimize_lgbm(X_tr, y_tr, X_val, y_val,
                           pos_weight, n_trials=OPTUNA_TRIALS)
print("Best params:", best_params)

## Cell 9 — Patient-Level GroupKFold Cross-Validation
Validates on train+val combined using GroupKFold(subject_id). No patient leakage across folds.

In [ ]:
def patient_level_cv(
    X: pd.DataFrame, y: pd.Series,
    groups: pd.Series, params: Dict,
    n_folds: int = N_FOLDS,
) -> Tuple[float, float, List[float]]:
    """
    GroupKFold CV grouped by subject_id.
    Guarantees no patient's records appear in both train and validation folds.
    Returns (mean_auroc, std_auroc, per_fold_aurocs).
    """
    logger.info("Patient-level GroupKFold CV (%d folds) ...", n_folds)
    gkf    = GroupKFold(n_splits=n_folds)
    scores = []

    for fold, (tr_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
        X_tr_cv  = X.iloc[tr_idx].values.astype(np.float32)
        y_tr_cv  = y.iloc[tr_idx].values
        X_val_cv = X.iloc[val_idx].values.astype(np.float32)
        y_val_cv = y.iloc[val_idx].values

        m = lgb.LGBMClassifier(**params)
        m.fit(
            X_tr_cv, y_tr_cv,
            eval_set=[(X_val_cv, y_val_cv)],
            callbacks=[lgb.early_stopping(50, verbose=False),
                       lgb.log_evaluation(-1)],
        )
        auc = roc_auc_score(y_val_cv, m.predict_proba(X_val_cv)[:, 1])
        scores.append(auc)
        logger.info(
            "  Fold %d AUROC: %.4f  (patients: %d train / %d val)",
            fold + 1, auc,
            groups.iloc[tr_idx].nunique(),
            groups.iloc[val_idx].nunique(),
        )
        del m, X_tr_cv, X_val_cv; gc.collect()

    mean, std = float(np.mean(scores)), float(np.std(scores))
    logger.info("CV AUROC: %.4f ± %.4f", mean, std)
    return mean, std, scores


In [ ]:
X_tv      = X[train_mask | val_mask]
y_tv      = y[train_mask | val_mask]
groups_tv = groups[train_mask | val_mask]
cv_mean, cv_std, cv_scores = patient_level_cv(X_tv, y_tv, groups_tv, best_params)
print(f"CV AUROC: {cv_mean:.4f} +/- {cv_std:.4f}")
print(f"Per-fold: {[round(s,4) for s in cv_scores]}")

## Cell 10 — Ensemble: LightGBM + XGBoost

In [ ]:
def _has_cuda() -> bool:
    try:
        import torch
        return torch.cuda.is_available()
    except ImportError:
        try:
            import subprocess
            result = subprocess.run(["nvidia-smi"], capture_output=True)
            return result.returncode == 0
        except Exception:
            return False


def build_ensemble(
    X_tr: np.ndarray, y_tr: np.ndarray,
    X_val: np.ndarray, y_val: np.ndarray,
    params: Dict, pos_weight: float,
) -> List[Tuple[str, object]]:
    """
    Trains LightGBM + XGBoost on the same training data.
    Returns list of (name, model) tuples for meta-learner stacking.
    """
    models: List[Tuple[str, object]] = []

    # -- LightGBM --------------------------------------------------------------
    logger.info("Training LightGBM ...")
    lgbm_model = lgb.LGBMClassifier(**params)
    lgbm_model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(100, verbose=False),
                   lgb.log_evaluation(-1)],
    )
    auc_lgbm = roc_auc_score(y_val, lgbm_model.predict_proba(X_val)[:, 1])
    logger.info("LightGBM val AUROC: %.4f", auc_lgbm)
    models.append(("lgbm", lgbm_model))

    # -- XGBoost ---------------------------------------------------------------
    if ENABLE_STACK:
        try:
            import xgboost as xgb
            logger.info("Training XGBoost ...")
            xgb_params = {
                "n_estimators":     1500,
                "learning_rate":    params.get("learning_rate", 0.05),
                "max_depth":        min(int(params.get("max_depth", 8)), 10),
                "subsample":        params.get("subsample", 0.8),
                "colsample_bytree": params.get("colsample_bytree", 0.8),
                "reg_alpha":        params.get("reg_alpha", 0.1),
                "reg_lambda":       params.get("reg_lambda", 1.0),
                "scale_pos_weight": pos_weight,
                "tree_method":      "hist",          # modern API (gpu_hist removed)
                "device":           "cuda" if _has_cuda() else "cpu",
                "eval_metric":      "auc",
                "random_state":     RANDOM_STATE,
                "n_jobs":           -1,
                "verbosity":        0,
            }
            # Handle both XGBoost < 2.0 (early_stopping in fit) and >= 2.0 (constructor)
            try:
                xgb_model = xgb.XGBClassifier(**xgb_params, early_stopping_rounds=50)
                xgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
            except TypeError:
                xgb_model = xgb.XGBClassifier(**xgb_params)
                xgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
                              early_stopping_rounds=50, verbose=False)
            auc_xgb = roc_auc_score(y_val, xgb_model.predict_proba(X_val)[:, 1])
            logger.info("XGBoost val AUROC: %.4f", auc_xgb)
            models.append(("xgb", xgb_model))
        except ImportError:
            logger.warning("xgboost not installed — skipping. pip install xgboost")
        except Exception as e:
            logger.warning("XGBoost failed (%s) — LightGBM only.", e)

    return models


def ensemble_predict(models: List[Tuple[str, object]], X: np.ndarray) -> np.ndarray:
    """Simple mean of model probabilities."""
    return np.stack([m.predict_proba(X)[:, 1] for _, m in models], axis=1).mean(axis=1)



In [ ]:
models = build_ensemble(X_tr, y_tr, X_val, y_val, best_params, pos_weight)
print(f"Ensemble members: {[name for name, _ in models]}")

## Cell 11 — Meta-Learner Stacking
Logistic Regression trained on val-set OOF predictions from LightGBM + XGBoost.

In [ ]:
def fit_meta_learner(
    models: List[Tuple[str, object]],
    X_val: np.ndarray, y_val: np.ndarray,
    X_te: np.ndarray,
) -> Tuple[Optional[LogisticRegression], np.ndarray]:
    """
    Trains Logistic Regression on val-set OOF predictions from each base model.
    Test probabilities come from the stacked meta prediction.
    """
    if len(models) < 2:
        return None, ensemble_predict(models, X_te)

    logger.info("Fitting meta-learner ...")
    val_stack = np.column_stack([m.predict_proba(X_val)[:, 1] for _, m in models])
    te_stack  = np.column_stack([m.predict_proba(X_te)[:, 1]  for _, m in models])

    meta = LogisticRegression(C=1.0, max_iter=1000, random_state=RANDOM_STATE)
    meta.fit(val_stack, y_val)

    auc_meta = roc_auc_score(y_val, meta.predict_proba(val_stack)[:, 1])
    logger.info("Meta-learner val AUROC: %.4f", auc_meta)
    return meta, meta.predict_proba(te_stack)[:, 1]



In [ ]:
if ENABLE_STACK and len(models) > 1:
    meta, test_probs_raw = fit_meta_learner(models, X_val, y_val, X_te)
    val_probs_for_cal = meta.predict_proba(
        np.column_stack([m.predict_proba(X_val)[:,1] for _,m in models])
    )[:,1]
else:
    meta = None
    test_probs_raw    = np.mean([m.predict_proba(X_te)[:,1] for _,m in models], axis=0)
    val_probs_for_cal = np.mean([m.predict_proba(X_val)[:,1] for _,m in models], axis=0)

print(f"Meta-learner: {'enabled' if meta else 'disabled (single model)'}")

## Cell 12 — Isotonic Calibration + Threshold Optimisation

In [ ]:
def calibrate(
    val_probs: np.ndarray, y_val: np.ndarray,
    test_probs: np.ndarray,
) -> Tuple[IsotonicRegression, np.ndarray]:
    """Isotonic regression calibration. Fitted on val, applied to test."""
    cal = IsotonicRegression(out_of_bounds="clip")
    cal.fit(val_probs, y_val)
    ece_before = _ece(val_probs, y_val)
    ece_after  = _ece(cal.predict(val_probs), y_val)
    logger.info("ECE before calibration: %.4f -> after: %.4f", ece_before, ece_after)
    return cal, cal.predict(test_probs).astype(np.float32)


def _ece(probs: np.ndarray, y: np.ndarray, n_bins: int = 10) -> float:
    bins, ece, total = np.linspace(0, 1, n_bins + 1), 0.0, len(y)
    for i in range(n_bins):
        mask = (probs >= bins[i]) & (probs < bins[i + 1])
        if mask.sum() == 0:
            continue
        ece += (mask.sum() / total) * abs(float(y[mask].mean()) - float(probs[mask].mean()))
    return ece


# ==============================================================================
# 8. THRESHOLD OPTIMISATION
# ==============================================================================

def find_best_threshold(
    val_probs: np.ndarray, y_val: np.ndarray,
    strategy: str = "f1",
) -> float:
    """
    Finds the optimal decision threshold on the VALIDATION SET only.
    Never uses test data — avoids look-ahead bias.

    strategy options
    ----------------
    "f1"       : maximise F1 on readmit=1 class (recommended default)
    "recall80" : highest precision while keeping recall ≥ 0.80
                 (clinical: catch at least 80% of readmissions)
    "j"        : Youden's J = sensitivity + specificity − 1
    """
    thresholds = np.arange(0.05, 0.70, 0.005)

    if strategy == "f1":
        scores = [f1_score(y_val, val_probs >= t, zero_division=0) for t in thresholds]
        best   = float(thresholds[np.argmax(scores)])

    elif strategy == "recall80":
        best = 0.50
        for t in sorted(thresholds, reverse=True):
            preds = (val_probs >= t).astype(int)
            rec   = preds[y_val == 1].mean() if (y_val == 1).sum() > 0 else 0.0
            if rec >= 0.80:
                best = float(t)
                break

    elif strategy == "j":
        fpr, tpr, thresh = roc_curve(y_val, val_probs)
        best = float(thresh[np.argmax(tpr - fpr)])

    else:
        best = 0.50

    logger.info("Threshold strategy='%s' -> best threshold=%.3f", strategy, best)
    return best


# ==============================================================================
# 9. PLOTS
# ==============================================================================


In [ ]:
calibrator, test_probs_cal = calibrate(val_probs_for_cal, y_val, test_probs_raw)
best_threshold = find_best_threshold(val_probs_for_cal, y_val, strategy='recall80')
print(f"Best threshold: {best_threshold:.3f}")

## Cell 13 — Final Test Metrics

In [ ]:
auc_raw  = roc_auc_score(y_te, test_probs_raw)
auc_cal  = roc_auc_score(y_te, test_probs_cal)
auprc    = average_precision_score(y_te, test_probs_cal)
brier    = brier_score_loss(y_te, test_probs_cal)
ll       = log_loss(y_te, test_probs_cal)

test_preds = (test_probs_raw >= best_threshold).astype(int)
report     = classification_report(y_te, test_preds, output_dict=True, zero_division=0)
recall_pos = report.get("1", {}).get("recall",    0.0)
prec_pos   = report.get("1", {}).get("precision", 0.0)
f1_pos     = report.get("1", {}).get("f1-score",  0.0)

print("=" * 55)
print("FINAL TEST RESULTS")
print(f"  AUROC (raw):          {auc_raw:.4f}")
print(f"  AUROC (calibrated):   {auc_cal:.4f}")
print(f"  AUPRC:                {auprc:.4f}")
print(f"  Brier Score:          {brier:.4f}")
print(f"  Log Loss:             {ll:.4f}")
print(f"  CV AUROC:             {cv_mean:.4f} +/- {cv_std:.4f}")
print(f"  Best threshold:       {best_threshold:.3f}")
print(f"  Recall  (readmit=1):  {recall_pos:.4f}")
print(f"  Precision(readmit=1): {prec_pos:.4f}")
print(f"  F1      (readmit=1):  {f1_pos:.4f}")
print("=" * 55)

## Cell 14 — Ablation Study
Compares AUROC for: fused (tabular + embeddings), tabular-only, embeddings-only.

In [ ]:
def run_ablation(
    X_tr: pd.DataFrame, y_tr: pd.Series,
    X_te: pd.DataFrame, y_te: pd.Series,
    params: Dict,
) -> Dict:
    """AUROC comparison: fused vs tabular-only vs embeddings-only."""
    tab_cols = [c for c in X_tr.columns if not c.startswith("ct5_")]
    emb_cols = [c for c in X_tr.columns if c.startswith("ct5_")]
    results  = {}

    for name, cols in [
        ("fused",        list(X_tr.columns)),
        ("tabular_only", tab_cols),
        ("text_only",    emb_cols or None),
    ]:
        if not cols:
            continue
        m = lgb.LGBMClassifier(**params)
        m.fit(
            X_tr[cols].values, y_tr.values,
            eval_set=[(X_te[cols].values, y_te.values)],
            callbacks=[lgb.early_stopping(50, verbose=False),
                       lgb.log_evaluation(-1)],
        )
        auc = roc_auc_score(y_te.values, m.predict_proba(X_te[cols].values)[:, 1])
        results[name] = round(float(auc), 4)
        logger.info("  Ablation %-14s: %.4f", name, auc)
        del m; gc.collect()

    return results


In [ ]:
X_df_tr  = pd.DataFrame(np.vstack([X_tr, X_val]), columns=features)
X_df_te  = pd.DataFrame(X_te, columns=features)
ablation = run_ablation(
    X_df_tr, pd.Series(np.concatenate([y_tr, y_val])),
    X_df_te, pd.Series(y_te), best_params,
)
print("Ablation results:", ablation)

## Cell 15 — SHAP Feature Importance

In [ ]:
def compute_shap(model, X_sample: pd.DataFrame) -> None:
    try:
        logger.info("Computing SHAP values (%d samples) ...", len(X_sample))
        exp = shap.TreeExplainer(model)
        sv  = exp.shap_values(X_sample)
        if isinstance(sv, list):
            sv = sv[1]
        plt.figure(figsize=(10, 8))
        shap.summary_plot(sv, X_sample, max_display=30, show=False)
        plt.tight_layout()
        plt.savefig(os.path.join(FIGURES_DIR, "shap_summary.png"), dpi=150,
                    bbox_inches="tight")
        plt.close()
        logger.info("SHAP plot saved.")
    except Exception as e:
        logger.warning("SHAP failed: %s", e)



In [ ]:
primary = next((m for n, m in models if n == 'lgbm'), None)
if primary:
    n_shap = min(3000, len(X_te))
    idx    = np.random.RandomState(RANDOM_STATE).choice(len(X_te), n_shap, replace=False)
    compute_shap(primary, pd.DataFrame(X_te[idx], columns=features))
    # Display inline in Colab
    from IPython.display import Image
    Image(os.path.join(FIGURES_DIR, 'shap_summary.png'))

## Cell 16 — ROC / PR / Calibration / Threshold Plots

In [ ]:
def save_plots(
    y_te: np.ndarray,
    test_probs_raw: np.ndarray,
    test_probs_cal: np.ndarray,
    best_thresh: float,
) -> None:
    os.makedirs(FIGURES_DIR, exist_ok=True)

    # -- ROC + Precision-Recall ------------------------------------------------
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    fpr, tpr, _ = roc_curve(y_te, test_probs_cal)
    axes[0].plot(fpr, tpr, lw=2,
                 label=f"AUROC = {roc_auc_score(y_te, test_probs_cal):.4f}")
    axes[0].plot([0, 1], [0, 1], "k--", lw=1)
    axes[0].set(xlabel="False Positive Rate", ylabel="True Positive Rate",
                title="ROC Curve")
    axes[0].legend(loc="lower right")

    prec, rec, _ = precision_recall_curve(y_te, test_probs_cal)
    axes[1].plot(rec, prec, lw=2,
                 label=f"AUPRC = {average_precision_score(y_te, test_probs_cal):.4f}")
    axes[1].axhline(y_te.mean(), color="gray", linestyle="--",
                    label=f"Baseline = {y_te.mean():.3f}")
    axes[1].set(xlabel="Recall", ylabel="Precision", title="Precision-Recall Curve")
    axes[1].legend(loc="upper right")

    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, "roc_pr_curve.png"), dpi=150)
    plt.close()

    # -- Calibration curve -----------------------------------------------------
    fig, ax = plt.subplots(figsize=(6, 6))
    for label, probs in [("Raw", test_probs_raw), ("Calibrated", test_probs_cal)]:
        fp, mp = _calibration_bins(probs, y_te)
        ax.plot(mp, fp, "s-", label=label)
    ax.plot([0, 1], [0, 1], "k--", label="Perfect")
    ax.set(xlabel="Mean Predicted Probability", ylabel="Fraction Positives",
           title="Calibration (Reliability Diagram)")
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, "calibration_curve.png"), dpi=150)
    plt.close()

    # -- Threshold analysis -----------------------------------------------------
    thresholds = np.arange(0.05, 0.70, 0.01)
    f1s, recs, precs = [], [], []
    for t in thresholds:
        preds = (test_probs_cal >= t).astype(int)
        f1s.append(f1_score(y_te, preds, zero_division=0))
        tp = int(((preds == 1) & (y_te == 1)).sum())
        fp = int(((preds == 1) & (y_te == 0)).sum())
        fn = int(((preds == 0) & (y_te == 1)).sum())
        recs.append(tp / max(tp + fn, 1))
        precs.append(tp / max(tp + fp, 1))

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(thresholds, f1s,   label="F1 (readmit=1)", lw=2)
    ax.plot(thresholds, recs,  label="Recall (readmit=1)", lw=2)
    ax.plot(thresholds, precs, label="Precision (readmit=1)", lw=2)
    ax.axvline(best_thresh, color="red", linestyle="--",
               label=f"Best threshold = {best_thresh:.3f}")
    ax.set(xlabel="Decision Threshold", ylabel="Score",
           title="Threshold Analysis — Readmit=1 Class", ylim=[0, 1])
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, "threshold_analysis.png"), dpi=150)
    plt.close()

    logger.info("Plots saved -> %s", FIGURES_DIR)


def _calibration_bins(probs, y, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    fp, mp = [], []
    for i in range(n_bins):
        mask = (probs >= bins[i]) & (probs < bins[i + 1])
        if mask.sum() == 0:
            continue
        fp.append(float(y[mask].mean()))
        mp.append(float(probs[mask].mean()))
    return np.array(fp), np.array(mp)



In [ ]:
save_plots(y_te, test_probs_raw, test_probs_cal, best_threshold)

# Display all plots inline
from IPython.display import Image, display
for fname in ['roc_pr_curve.png', 'calibration_curve.png', 'threshold_analysis.png']:
    path = os.path.join(FIGURES_DIR, fname)
    if os.path.exists(path):
        print(f"--- {fname} ---")
        display(Image(path))

## Cell 17 — Save Model + Results to Google Drive

In [ ]:
import os

# Compute feature means for 08_predict.py defaults
try:
    X_all_np = np.vstack([X_tr, X_val])
    feature_means = {
        feat: float(np.nanmean(X_all_np[:, i]))
        for i, feat in enumerate(features)
    }
    print(f"Feature means computed: {len(feature_means)} features")
except Exception as e:
    feature_means = {}
    print(f"Could not compute feature means: {e}")

# Save model pkl
os.makedirs(MODELS_DIR, exist_ok=True)
joblib.dump({
    'models':         models,
    'meta':           meta,
    'calibrator':     calibrator,
    'features':       features,
    'best_params':    best_params,
    'best_threshold': best_threshold,
    'feature_means':  feature_means,
    'timestamp':      datetime.now().isoformat(),
}, MAIN_MODEL_PKL)
print(f"Model saved -> {MAIN_MODEL_PKL}")

# Save predictions CSV
pred_df = pd.DataFrame({
    'y_true':   y_te,
    'prob_raw': test_probs_raw.round(6),
    'prob_cal': test_probs_cal.round(6),
    'pred':     test_preds,
})
pred_path = os.path.join(RESULTS_DIR, 'test_predictions.csv')
pred_df.to_csv(pred_path, index=False)
print(f"Predictions saved -> {pred_path}")

# Save training report JSON
results = {
    'auroc_raw':          round(float(auc_raw),   4),
    'auroc_calibrated':   round(float(auc_cal),   4),
    'auprc':              round(float(auprc),      4),
    'brier_score':        round(float(brier),      4),
    'log_loss':           round(float(ll),         4),
    'cv_auroc_mean':      round(float(cv_mean),    4),
    'cv_auroc_std':       round(float(cv_std),     4),
    'best_threshold':     round(float(best_threshold), 3),
    'recall_readmit1':    round(float(recall_pos), 4),
    'precision_readmit1': round(float(prec_pos),   4),
    'f1_readmit1':        round(float(f1_pos),     4),
    'ablation':           ablation,
    'n_models':           len(models),
    'n_features':         len(features),
    'train_size':         int(len(y_tr)),
    'val_size':           int(len(y_val)),
    'test_size':          int(len(y_te)),
    'best_params':        {k: str(v) if not isinstance(v,(int,float,bool,str,type(None))) else v
                           for k,v in best_params.items()},
    'timestamp':          datetime.now().isoformat(),
}
report_path = os.path.join(RESULTS_DIR, 'training_report.json')
with open(report_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f"Report saved -> {report_path}")

print("\n=== All outputs saved to Google Drive ===")

## Cell 18 — Quick Smoke Test (Optional)
Run this cell **instead of cells 8–17** to validate the full pipeline in ~15 minutes using only 5 Optuna trials and 50k training rows.


In [ ]:
# ── SMOKE TEST — validates pipeline end-to-end quickly ──────────────────
SMOKE_ROWS   = 50_000
SMOKE_TRIALS = 5

logger.info("SMOKE TEST: %d rows, %d Optuna trials", SMOKE_ROWS, SMOKE_TRIALS)

# Subsample
idx_sample = np.random.RandomState(RANDOM_STATE).choice(len(X_tr), min(SMOKE_ROWS, len(X_tr)), replace=False)
X_tr_s = X_tr[idx_sample]; y_tr_s = y_tr[idx_sample]

bp = optimize_lgbm(X_tr_s, y_tr_s, X_val, y_val, pos_weight, n_trials=SMOKE_TRIALS)
m  = lgb.LGBMClassifier(**bp)
m.fit(X_tr_s, y_tr_s, eval_set=[(X_val, y_val)],
      callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
auc = roc_auc_score(y_te, m.predict_proba(X_te)[:,1])
print(f"Smoke test AUROC: {auc:.4f}  (expected ~0.75+ for tabular-only)")
print("Pipeline is working correctly." if auc > 0.6 else "WARNING: Low AUROC — check data loading.")